# Task B. Symmetric Cipher (10 marks)

In this task, we aim to let students experience in using the an implementation of symmetric ciphers based on **AES** and develop further understanding of some important features of modern symmetric cipher, e.g., the avalanche effect. Refer to https://cryptography.io/en/latest/hazmat/primitives/symmetric-encryption/#cryptography.hazmat.primitives.ciphers.algorithms.AES for detailed documentation of AES and https://cryptography.io/en/latest/fernet/ for Fernet. 

In the following source code, we have implemented two functions to produce keys from a given string. You can use this function to create your key to use AES with the ECB mode and Fernet functions. We have demonstrated examples of using AES and Fernet to encrypt and decrypt some texts. **Please pay attention to the encoding/decoding that achieves conversion between a string and a binary string which is the input or output of a AES/Fernet function**. 

In [1]:
from cryptography.hazmat.primitives.ciphers import Cipher, algorithms, modes
from cryptography.hazmat.primitives.ciphers.algorithms import AES
from cryptography.hazmat.backends import default_backend
from cryptography.hazmat.primitives import padding
import os
import sys

# if you get an error on the above line, you might need to run 
# pip install INSERT_LIBRARY_NAME or install the library another way.

#=====================================================
# Please do NOT modify the following code, but you are more than welcome to understand the code in detail

# This function generates a key from a given string representing a student's ID for the AES cipher. 
# The base64 encoding changes 3 bytes into 4 bytes. Resultant alphabet contains letters, numbers, and three symbols +, /, and = (for padding)
def generate_key_from_string(key_string="please don't use the default"):
    if(len(key_string)<24):  # Because the key length of AES can be 128 bits, 192 bits, and 256 bits. We use the 192 bit here, i.e., 24 bytes.
       key_string = str(key_string + "abcdefghijklmnopqrstuvwx")
    key_string = key_string[0:24]
    key_string_bytes = str(key_string).encode("utf-8")
    return key_string_bytes


# Examples
# Generate a key from a string
key = generate_key_from_string(key_string="12345678")
print("Example key is: ", key)
print("Length of the encoded key is: %d bytes" % len(key)) # The length should be a multiple of 4.
print("")

# Now we use the ECB mode with AES to perform block cipher operations on the following message.
# For more details about AES APIs, please refer to: https://cryptography.io/en/latest/hazmat/primitives/symmetric-encryption/#cryptography.hazmat.primitives.ciphers.algorithms.AES

# Plain text
plain_text = "Hellow World! Welcome to COMP2300/6300!"

# Convert the string to binary string
plain_text_bytes = bytes(plain_text, "utf-8")
print("Example plaintext byte string: ", plain_text_bytes)

# Convert the byte string to binary string
plain_text_binary_chars = []
for i in plain_text_bytes:
    plain_text_binary_chars.append(format(i, '08b'))
plain_text_binary_string = ''.join(plain_text_binary_chars)
print("Binary string for the plaintext: ", plain_text_binary_string)
print("Plaintext binary string length: ", len(plain_text_binary_string))

# At first, we need to pad the plaintext to make its length a multiple of the block size (16 bytes per block for AES)
padder = padding.PKCS7(AES.block_size).padder()
padded_plaintext = padder.update(plain_text_bytes)+padder.finalize()

# Create a cipher instance with AES in CBC mode.
# An intialisation vector is needed
iv = os.urandom(16)
cipher = Cipher(AES(key), modes.CBC(iv), backend=default_backend())

# Encrypt the plaintext
encryptor = cipher.encryptor() # encryptor here is a CipherContext object: https://cryptography.io/en/latest/hazmat/primitives/symmetric-encryption/#cryptography.hazmat.primitives.ciphers.CipherContext
cipher_text = encryptor.update(padded_plaintext)+encryptor.finalize()
print("Example ciphertext is: ", cipher_text)

# Binary string of the cipher text
cipher_text_binary_chars = []
for i in cipher_text:
    cipher_text_binary_chars.append(format(i, '08b'))
cipher_text_binary_string = ''.join(cipher_text_binary_chars)
print("Binary string for ciphertext: ", cipher_text_binary_string)
print("Ciphertext binary string length: ", len(cipher_text_binary_string))

# Decrypt the ciphertext
decryptor = cipher.decryptor()
decrypted_padded_text = decryptor.update(cipher_text) + decryptor.finalize()

# Remove the padding after decryption
unpadder = padding.PKCS7(AES.block_size).unpadder()
decrypted_text_bytes = unpadder.update(decrypted_padded_text) + unpadder.finalize()

# Convert the bytes back into string
decrypted_text = decrypted_text_bytes.decode("utf-8")
print("Example decrypted text is: ", decrypted_text)
print("")

Example key is:  b'12345678abcdefghijklmnop'
Length of the encoded key is: 24 bytes

Example plaintext byte string:  b'Hellow World! Welcome to COMP2300/6300!'
Binary string for the plaintext:  010010000110010101101100011011000110111101110111001000000101011101101111011100100110110001100100001000010010000001010111011001010110110001100011011011110110110101100101001000000111010001101111001000000100001101001111010011010101000000110010001100110011000000110000001011110011011000110011001100000011000000100001
Plaintext binary string length:  312
Example ciphertext is:  b'*\xdeiYq\x04s\x9b\xdd\x10\xf9\xff\x82\xd9\xfcVu\x01\xf0\xd9GY.\x90ew{`\x0c\x1f\x8bH6\x18\x0e\x0f\x1a[\x1e\x90>\x0c\x90\r\x8csLS'
Binary string for ciphertext:  001010101101111001101001010110010111000100000100011100111001101111011101000100001111100111111111100000101101100111111100010101100111010100000001111100001101100101000111010110010010111010010000011001010111011101111011011000000000110000011111100010110100100000110110000110

## Task B.1 (5 marks)

You are required to use AES to experience several techniques used in modern block ciphers, such as **padding** and the **avalanche effect** from the **diffusion** and **confusion** operations in modern block ciphers. To observe the avalanche effect, you need to write code to calculate the percentage of the bit difference between the ciphertexts from two plaintexts of a minor difference. Please carefully go through the follwoing documentations to understand the AES related APIs and answer the questions correctly: https://cryptography.io/en/latest/hazmat/primitives/symmetric-encryption/#cryptography.hazmat.primitives.ciphers.algorithms.AES.

**Question B.1.1 (0.5 mark)**:  In the example demonstrated above, do the plaintext and the ciphertext have the same length as we expect? Please explain the observation in detail.

**Answer**:

**Question B.1.2 (1 mark)**: Modify the binary string of the **plaintext** by 1 bit to create a variant of the origianl plaintext (i.e., "Hellow World! Welcome to COMP2300/6300!"). What is the modified plaintext?
* You decide which bit to change. You can change the bit directly on the binary string of the plaintext, or simply replace a character in the plaintext with a neighbouring character (e.g., '$'-->'%'). Please refer to Appendix A of Task A to get the binary coding of each printable character.

**Answer**:

**Question B.1.3 (0.5 mark)**: You are required to encrypt the modified plaintext with AES. The mode is still CBC. What is the ciphertext?

**Answer**:

**Question B.1.4 (1 mark)**: You are required to compare the ciphertext from the modified plaintext with the ciphertext from the original plaintext. What is the percentage of bit difference in the two pieces of ciphertext. Note that percetage of difference should be calculated **at the bit level**.

**Answer**:

**Question B.1.5 (1 mark)**: You are required to modify the binary string of the **ciphertext** by 1 bit to create a variant of the ciphertext from the origianl plaintext (i.e., "Hellow World! Welcome to COMP2300/6300!"). Then, you are required decryped the modified ciphertext. What is the plaintext of the modified ciphertext?
* This is different from Question B.1.2 where the **plaintext** is modified.
* You decide which bit to change.
* With modification on the ciphertext, it is possible that the decrypted plaintext bytes cannot be decoded by "utf-8". If this happens, please try to use the coding scheme "ISO-8859-1". "ISO-8859-1" uses a fixed 1 byte per character, while "utf-8" uses a variable number of bytes.

**Answer**:

**Task B.1 Source Code (1 mark)**

In [3]:
# TODO: Your code to answer the questions above.




## Task B.2 (3 marks)

In this task, we aim to let students experience in using different **operation modes** of the implementation of modern symmetric ciphers and develop further understanding of how these operation modes affect the security of the ciphers. The operation modes APIs and documentation can be found here: https://cryptography.io/en/latest/hazmat/primitives/symmetric-encryption/#module-cryptography.hazmat.primitives.ciphers.modes.

Now, let's change the operation mode from CBC used in the examples to a new operation mode **CTR**. Run AES again (but with CTR) to encrypt the same plaintext ("Hellow World! Welcome to COMP2300/6300!") and its variant you created in **Question B.1.2**.
* If an **initialisation vector (IV)** or **nonce** is necessary in an operation mode, you can use the *urandom()* funciton in the *os* module to generate it.

**Question B.2.1 (0.5 mark)**: For the given plaintext and its variant, what are the two corresponding ciphertexts produced by AES with CTR?

**Answer**:

**Question B.2.2 (0.5 mark)**: What is the percentage of bit difference in the two pieces of ciphertext? 
* Note that percetage of difference should be calculated at the bit level. 

**Answer**:

**Question B.2.3 (1 mark)**: Is there a significant difference between the bit difference percentage for CTR and the bit difference percentage for CBC (obtained from **Question B.1.4**)? Please explain why.

**Answer**:

**Task B.2 Source Code (1 mark)**

In [5]:
# TODO: Your code to answer the questions above.



## Task B.3 (2 marks)

In this task, you are requested to call Fernet APIs to decrypt a ciphertext and encrypt a plaintext. The cipher used to encrypt the file is **Fernet**, which uses AES in CBC mode with a 128-bit key for encryption and PKCS7 for padding. The details of Fernet can be found here: https://cryptography.io/en/latest/fernet/. Following are some example of using Fernet APIs.

In [7]:
#===================================================
# Fernet examples
from cryptography.fernet import Fernet # for encryption and decryption
import base64

# This function generates a key from a given string representing a student's ID
# The base64 encoding changes 3 bytes into 4 bytes. Resultant alphabet contains letters, numbers, and three symbols +, /, and = (for padding)
def generate_key_from_string_for_fernet(key_string="please don't use the default"):
    if(len(key_string)<32):
       key_string = str(key_string + "abcdefghijklmnopqrstuvwxyz012345")
    key_string = key_string[0:32]
    key_string_bytes = str(key_string).encode("utf-8")
    key = base64.urlsafe_b64encode(key_string_bytes) 
    return key

# Generate a key from a string
key = generate_key_from_string_for_fernet(key_string="12345678")

# Create a fernet with the key
fernet = Fernet(key)

# Plain text
plain_text_fernet = b'Hellow World, COMP2300/6300!'

# Encrypt the plain text and print the cipher text
cipher_text_fernet = fernet.encrypt(plain_text_fernet)
print("Example ciphertext from Fernet is: ", cipher_text_fernet)

# Decrypt the cipher text
decrypted_text_bytes_fernet = fernet.decrypt(cipher_text_fernet)
print("Example decrypted text is: ", decrypted_text_bytes_fernet)
print("")

Example ciphertext from Fernet is:  b'gAAAAABpqOcKY3G1Mgz8_7qkGK4KtiqVe_Fk2fgqhFzltnNVdBVQs9ImmU-XYEWPbfyxUrBhoHWVY1uKWrlx4j5UWBAeVo6ERzhisPFx-6b5bT-x6SXYJ6o='
Example decrypted text is:  b'Hellow World, COMP2300/6300!'



**Question B.3.1 (0.5 mark)**: You are required to perform decryption via fernet. There are two txt files containing the cipher text and you just need to choose one of them as the input for the decryption function. 
* For Linux/Mac users, please use the file “mac_linux_data_encrypted.txt”,
* For the Windows users, please use the file “windows_data_encrypted.txt”.
* The key information is from the answer to **Question A.3.5** in Task A.
* The decrypted text should be saved in a file with the file name “\<YourStudentID\>_Task-B-3_plaintext.txt” for submission, which will be submitted for evaluation and marking.
*  The decrypted text should be readable in English by human.

Have you successfully decrypt the given ciphertext?

**Answer**:

**Question B.3.2 (0.5 mark)**: You are required to encrypt the plaintext deciphered in answering **Question B.3.1**,  with a **different key**. 
* The key information is from the answer to **Question A.3.5** in Task A.
* The ciphertext should be stored in a file which will be submitted for evaluation and marking (The markers will check if it is able to be decrypted with your given key).
* The ciphertext file should be named with the format "\<YourStudentID\>_Task-B-3_ciphertext.txt", where \<YourStudentID\> should be replaced by your own student ID.

Have you successfully encrypted the plaintext with the required key?

**Answer**:

**Task B.3 Source Code (1 mark)**

In [9]:
# TODO: Your code to decrypt the cipher text above

